# Qwen Chat Image Studio — Colab L4

A conversational image generator/editor designed around an NVIDIA L4 (24 GB).

**Workflow**
1. Type a natural request: `a cozy cyberpunk ramen shop in the rain`
2. The chat model expands it into a structured image prompt.
3. The first turn uses **Qwen-Image-2512** for real text-to-image generation.
4. Follow-ups such as `make the jacket red` use the **previous image automatically** with **Qwen-Image-Edit-2511**.
5. The latest image is kept in memory, so you do **not** re-upload it after every edit.
6. The conversation also keeps a compact scene state so edits preserve important details.

This notebook deliberately does **not** fake first-generation by sending a blank canvas to an image editor.

## Model choices

- **Qwen-Image-2512**: first-turn text-to-image generation.
- **Qwen-Image-Edit-2511**: subsequent image editing and multi-turn consistency.
- **Qwen2.5-3B-Instruct**: lightweight conversational/orchestration layer, loaded 4-bit.

The official Qwen repository documents Qwen-Image-2512 for text-to-image and Qwen-Image-Edit-2511 for editing. The official edit checkpoint is large, so this notebook supports an optional community FP8 + Lightning 8-step transformer path for the L4. The FP8 path is optional; if its repository format changes, switch `USE_LIGHTNING_FP8` to `False` and use the official BF16 checkpoint with offload.

In [ ]:
#@title 1. Install dependencies
!pip -q install -U git+https://github.com/huggingface/diffusers.git
!pip -q install -U transformers accelerate bitsandbytes sentencepiece safetensors gradio huggingface_hub torchao

In [ ]:
#@title 2. Check GPU
import torch, os, gc, platform

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"VRAM: {p.total_memory/1024**3:.1f} GB")
assert torch.cuda.is_available(), "Enable a CUDA GPU in Colab (L4 recommended)."

In [ ]:
#@title 3. Optional Hugging Face login
# If a model asks for authentication in your environment, uncomment:
# from huggingface_hub import login
# login()

In [ ]:
#@title 4. Conversation model (Qwen2.5-3B-Instruct, 4-bit)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

CHAT_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

chat_tokenizer = AutoTokenizer.from_pretrained(CHAT_MODEL_ID)
chat_model = AutoModelForCausalLM.from_pretrained(
    CHAT_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
chat_model.eval()
print("Chat model ready.")

In [ ]:
#@title 5. Prompt/orchestration layer
import json, re

SYSTEM_PROMPT = r'''
You are the orchestration brain for a conversational image generator/editor.

Return ONLY valid JSON with this schema:
{
  "action": "generate" | "edit",
  "image_prompt": "detailed instruction for the image model",
  "scene_state": {
    "subject": "...",
    "appearance": "...",
    "clothing": "...",
    "environment": "...",
    "lighting": "...",
    "camera": "...",
    "style": "...",
    "text_in_image": "..."
  }
}

Rules:
- On the first user request, action must be "generate".
- On later turns, action must be "edit" unless the user clearly asks for a completely new image.
- Preserve everything not explicitly changed.
- For edits, image_prompt must describe the requested change plus the important unchanged scene details.
- Never say "use the previous image" as the whole prompt. State exactly what should be preserved and changed.
- If the request is vague, infer sensible visual details instead of asking unnecessary questions.
- Keep scene_state compact but useful.
'''

def ask_orchestrator(user_text, turns, scene_state):
    history = turns[-8:]
    payload = {
        "current_scene_state": scene_state,
        "recent_conversation": history,
        "new_user_request": user_text,
    }
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": json.dumps(payload, ensure_ascii=False),
        },
    ]
    
    # IMPORTANT:
    # apply_chat_template returns a BatchEncoding when return_dict=True.
    # Do NOT pass the BatchEncoding itself to generate().
    inputs = chat_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    
    # Cleanly move tensor fields to the model device
    inputs = inputs.to(chat_model.device)
    input_ids = inputs["input_ids"]
    
    with torch.inference_mode():
        output_ids = chat_model.generate(
            **inputs,
            max_new_tokens=700,
            do_sample=False,
        )
        
    # Remove the prompt tokens.
    generated_ids = output_ids[0, input_ids.shape[-1]:]
    text = chat_tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()
    
    # Extract JSON even if the model wraps it in ```json ... ```
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError(
            "Orchestrator did not return valid JSON:\n\n" + text
        )
    return json.loads(match.group(0))


## 6. Image models

The first turn uses the official **Qwen-Image-2512** text-to-image pipeline.

Follow-up edits use **Qwen-Image-Edit-2511**. For an L4, the notebook can use
`reb82/qwen-image-edit-2511-lightning-fp8`, which provides the edit transformer's
Lightning-8step LoRA already fused and FP8-quantized. Its model card says the
checkpoint is transformer-only and is intended to be combined with the base
Qwen-Image-Edit-2511 pipeline.

**Important L4 design:** the generation and editing pipelines are loaded lazily.
When switching from generation to editing, the generation pipeline is released
before the editor is loaded. This avoids trying to keep both large image models
resident simultaneously.

If the optional FP8 checkpoint fails to load in your Colab environment, the code
automatically falls back to the official BF16 Qwen-Image-Edit-2511 pipeline.

In [ ]:
#@title 6. Lazy-load the Qwen image pipelines (L4-safe)
import torch, gc
from diffusers import QwenImagePipeline, QwenImageEditPlusPipeline, QwenImageTransformer2DModel

GEN_MODEL_ID = "Qwen/Qwen-Image-2512"
EDIT_MODEL_ID = "Qwen/Qwen-Image-Edit-2511"

# This is the pre-quantized FP8 + Lightning-8step transformer.
# It is transformer-only; the base Qwen edit repo supplies the other components.
USE_LIGHTNING_FP8 = True
LIGHTNING_TRANSFORMER_ID = "reb82/qwen-image-edit-2511-lightning-fp8"

gen_pipe = None
edit_pipe = None
EDIT_STEPS = 40

def free_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def unload_image_pipes():
    global gen_pipe, edit_pipe
    gen_pipe = None
    edit_pipe = None
    free_cuda()

def load_generation_pipe():
    global gen_pipe, edit_pipe
    if gen_pipe is not None:
        return gen_pipe
    # Never keep both giant image pipelines resident.
    edit_pipe = None
    free_cuda()

    gen_pipe = QwenImagePipeline.from_pretrained(
        GEN_MODEL_ID,
        torch_dtype=torch.bfloat16,
    )
    try:
        gen_pipe.enable_model_cpu_offload()
    except Exception:
        gen_pipe.to("cuda")
    return gen_pipe

def load_edit_pipe():
    global gen_pipe, edit_pipe, EDIT_STEPS
    if edit_pipe is not None:
        return edit_pipe

    # Release the generation pipeline before loading the editor.
    gen_pipe = None
    free_cuda()

    if USE_LIGHTNING_FP8:
        try:
            transformer = QwenImageTransformer2DModel.from_pretrained(
                LIGHTNING_TRANSFORMER_ID,
                torch_dtype=torch.bfloat16,
                use_safetensors=False,
            )
            edit_pipe = QwenImageEditPlusPipeline.from_pretrained(
                EDIT_MODEL_ID,
                transformer=transformer,
                torch_dtype=torch.bfloat16,
            )
            EDIT_STEPS = 8
            print("Loaded pre-quantized FP8 + Lightning 8-step editor.")
        except Exception as e:
            print("FP8 editor failed; falling back to official BF16 editor.")
            print("Reason:", repr(e))
            free_cuda()
            edit_pipe = QwenImageEditPlusPipeline.from_pretrained(
                EDIT_MODEL_ID,
                torch_dtype=torch.bfloat16,
            )
            EDIT_STEPS = 40
    else:
        edit_pipe = QwenImageEditPlusPipeline.from_pretrained(
            EDIT_MODEL_ID,
            torch_dtype=torch.bfloat16,
        )
        EDIT_STEPS = 40

    try:
        edit_pipe.enable_model_cpu_offload()
    except Exception:
        edit_pipe.to("cuda")
    return edit_pipe

print("Pipelines are lazy-loaded. Only one large image pipeline is kept in memory at a time.")

In [ ]:
#@title 7. Generation/edit functions
from PIL import Image

ASPECTS = {
    "1:1": (1328, 1328),
    "16:9": (1664, 928),
    "9:16": (928, 1664),
    "4:3": (1472, 1104),
    "3:4": (1104, 1472),
    "3:2": (1584, 1056),
    "2:3": (1056, 1584),
}

def generate_image(prompt, aspect="1:1", seed=42):
    pipe = load_generation_pipe()
    w, h = ASPECTS[aspect]
    g = torch.Generator(device="cuda").manual_seed(int(seed))
    with torch.inference_mode():
        result = pipe(
            prompt=prompt,
            negative_prompt=" ",
            width=w,
            height=h,
            num_inference_steps=40,
            true_cfg_scale=4.0,
            generator=g,
        )
    return result.images[0]

def edit_image(image, prompt, seed=42):
    pipe = load_edit_pipe()
    g = torch.Generator(device="cuda").manual_seed(int(seed))
    with torch.inference_mode():
        result = pipe(
            image=[image],
            prompt=prompt,
            generator=g,
            true_cfg_scale=4.0,
            negative_prompt=" ",
            num_inference_steps=EDIT_STEPS,
            guidance_scale=1.0,
            num_images_per_prompt=1,
        )
    return result.images[0]

## 8. Conversational state

`current_image` is the important part: every successful generation/edit replaces it with the newest image.

That means:
- turn 1 → generate
- turn 2 → edit turn 1's image
- turn 3 → edit turn 2's image
- and so on

The user never has to upload the generated image again.

In [ ]:
#@title 8. Stateful chat engine
current_image = None
scene_state = {}
turns = []
last_prompt = ""

def reset_session():
    global current_image, scene_state, turns, last_prompt
    current_image = None
    scene_state = {}
    turns = []
    last_prompt = ""
    return None, [], "Session reset."

def run_turn(user_text, aspect="1:1", seed=42):
    global current_image, scene_state, turns, last_prompt

    user_text = (user_text or "").strip()
    if not user_text:
        return current_image, turns, "Type an image request."

    plan = ask_orchestrator(user_text, turns, scene_state)
    action = plan.get("action", "generate")
    prompt = plan.get("image_prompt", user_text)
    new_state = plan.get("scene_state", scene_state)

    # Safety for the first turn: always use real text-to-image generation.
    if current_image is None:
        action = "generate"

    if action == "generate":
        image = generate_image(prompt, aspect=aspect, seed=seed)
        mode = "generated"
    else:
        image = edit_image(current_image, prompt, seed=seed)
        mode = "edited"

    current_image = image
    scene_state = new_state
    last_prompt = prompt

    turns.append({
        "user": user_text,
        "action": mode,
        "image_prompt": prompt,
    })

    return current_image, turns, f"{mode.upper()}\n\n{prompt}"

In [ ]:
#@title 9. Optional: upload a starting image
from google.colab import files
from PIL import Image

def upload_start_image():
    global current_image, scene_state, turns, last_prompt
    uploaded = files.upload()
    if not uploaded:
        return
    name = next(iter(uploaded))
    current_image = Image.open(name).convert("RGB")
    scene_state = {"subject": "user-provided starting image"}
    turns = [{"user": "[uploaded image]", "action": "starting image"}]
    last_prompt = ""
    print("Starting image loaded. Your next message will edit it.")

## 10. Gradio UI

Run the next cell and use the chat box like:

> A stylish woman in a black leather jacket standing in Tokyo at night

Then:

> make the jacket red

Then:

> change the lighting to sunrise, keep everything else

Then:

> make it a wide cinematic shot

Each edit operates on the current image.

In [ ]:
#@title 10. Launch the conversational UI
import gradio as gr

def ui_turn(message, aspect, seed, chat_history):
    image, turns_out, status = run_turn(message, aspect, int(seed))
    chat_history = chat_history or []
    chat_history = chat_history + [(message, status)]
    return "", image, chat_history

def ui_reset():
    image, chat, status = reset_session()
    return image, chat, status

with gr.Blocks(title="Qwen Chat Image Studio") as demo:
    gr.Markdown("# Qwen Chat Image Studio")
    gr.Markdown("Natural-language generation + persistent conversational image editing.")

    with gr.Row():
        with gr.Column(scale=1):
            prompt_box = gr.Textbox(
                label="What do you want?",
                placeholder="e.g. a futuristic motorcycle in a rainy Tokyo alley",
                lines=3,
            )
            aspect = gr.Dropdown(
                choices=list(ASPECTS.keys()),
                value="1:1",
                label="Aspect ratio",
            )
            seed = gr.Number(value=42, precision=0, label="Seed")
            send = gr.Button("Generate / Edit", variant="primary")
            reset = gr.Button("Reset session")
            status = gr.Markdown()

        with gr.Column(scale=1):
            output = gr.Image(label="Current image", type="pil")
            chat = gr.Chatbot(label="Conversation", height=500)

    send.click(
        ui_turn,
        inputs=[prompt_box, aspect, seed, chat],
        outputs=[prompt_box, output, chat],
    )
    prompt_box.submit(
        ui_turn,
        inputs=[prompt_box, aspect, seed, chat],
        outputs=[prompt_box, output, chat],
    )
    reset.click(
        ui_reset,
        outputs=[output, chat, status],
    )

demo.launch(share=True, debug=True)

## Notes / troubleshooting

### L4 vs T4
Use the **L4 24 GB** runtime. The image checkpoints are much larger than GPU
memory, so the notebook uses CPU offload and lazy model switching.

### Why lazy loading matters
The notebook does **not** keep Qwen-Image-2512 and Qwen-Image-Edit-2511 loaded
simultaneously. It releases one before loading the other. This is important for
Colab RAM as well as VRAM.

### If the FP8 editor fails
Set:

```python
USE_LIGHTNING_FP8 = False
```

and rerun the image-pipeline cell. The notebook will use the official BF16
Qwen-Image-Edit-2511 pipeline with offload.

### Starting from your own image
Run `upload_start_image()` once. The next conversational request edits that
image, and every later edit uses the newest result automatically.

### Architecture
- Qwen2.5-3B-Instruct = conversational intent + scene-state manager
- Qwen-Image-2512 = genuine first-turn text-to-image
- Qwen-Image-Edit-2511 = persistent iterative editing

This is intentionally a two-stage image architecture rather than using an
image-edit model with a blank placeholder for first generation.